# Исследовательский анализ данных SOC (ML-трек)
**Датасет:** `iavd_incidents_raw_target.parquet`  
**Цель:** Выполнить разведочный анализ (EDA) по 7 блокам, проверить контрольные числа и сформулировать инсайты.

In [4]:
#!pip install -r requirements.txt

In [5]:
# Импорт библиотек и загрузка данных
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Путь к данным относительно папки ml/
df = pd.read_parquet("../data/iavd_incidents_raw_target.parquet")
print(f"Датасет успешно загружен. Размер: {df.shape}")

Датасет успешно загружен. Размер: (240295, 97)


In [15]:
print(df.columns.tolist())

['EntityId', 'Создан', 'Дата', 'Вердикт', 'Дата закрытия (факт)', 'Попытка эксплуатации', 'Тип инцидента', 'Список вредоносных команд', 'Критичность', 'Ответственный', 'Пиктограмма IRP', 'Прошлый ответственный', 'Типы угроз', 'Группа устранения', 'Рекомендации по плейбуку', 'Фаза реагирования', 'Теги', 'Источник инцидента', 'Вендор из сием', 'Организация', 'Наименование', 'suser', 'filePath', 'processParentPath', 'objectLinks', 'Ответсвенный по РП', 'Принятые меры', 'objectValue', 'Значение Атаки (группировка)', 'Решение', 'Алерт', 'Правило корреляции SIEM', 'Критичность инцидента в источнике', 'Полное имя', 'Статус инцидента из SIEM', 'Значение Инцидента (группировка)', 'Значение Инцидента (группировка по типу)', 'Файловый ресурс', 'Имя файла-IRP', 'Принятые меры из SIEM', 'Команда', 'CMDLINE', 'Конвейер SIEM', 'Описание инцидента', 'Hostname Цели', 'categoryHigh', 'categoryLow', 'categoryMain', 'commandName', 'duser', 'object', 'object.name', 'Привилегии аккаунта', 'processName', 'pr

## Блок 1. Знакомство со структурой данных

In [6]:
# Вопросы 1.1 - 1.4: Размерности, типы, пропуски
print("1.1 Размер:", df.shape)
print("1.2 Типы данных:\n", df.dtypes.value_counts())

missing = df.isna().mean().sort_values(ascending=False)
print("1.3 Топ-3 худших поля:\n", missing.head(3))
print("1.4 Полей без NaN:", (df.isna().mean() == 0).sum())

# Вопросы 1.5 - 1.7: Уникальные значения и мультизначные поля
print("1.5 Уникальных Вердикт:", df['Вердикт'].nunique())
print("1.6 Уникальных Критичность:", df['Критичность'].nunique())

# 1.7 Доля карточек с '|' в Значение Инцидента (группировка)
col_val = 'Значение Инцидента (группировка)'
has_pipe = df[col_val].dropna().str.contains(r'\|', regex=True).mean()
print(f"1.7 Доля непустых с '|': {has_pipe:.1%}")

# Вопросы 1.8 - 1.11: Разворот, дубликаты, числовой аналог
max_elements = df[col_val].dropna().str.split('~~||~~').str.len().max()
print(f"1.8 Макс элементов в списке: {max_elements}")
print("1.9 Дубликатов EntityId:", df['EntityId'].duplicated().sum())

# 1.10 Оценка строк после explode (берем срез для скорости, либо считаем точно)
total_elements = df[col_val].dropna().str.split('~~||~~').str.len().sum()
print(f"1.10 Строк после explode: ≈{total_elements/1e6:.1f} млн")

print("1.11 Числовой аналог Вердикт:", df['verdict'].dtype)

1.1 Размер: (240295, 97)
1.2 Типы данных:
 object                 77
datetime64[us, UTC]    11
float64                 7
int64                   2
Name: count, dtype: int64
1.3 Топ-3 худших поля:
 Ошибки                   0.999946
Типы угроз               0.997283
Прошлый ответственный    0.995443
dtype: float64
1.4 Полей без NaN: 8
1.5 Уникальных Вердикт: 7
1.6 Уникальных Критичность: 5
1.7 Доля непустых с '|': 100.0%
1.8 Макс элементов в списке: 55353
1.9 Дубликатов EntityId: 0
1.10 Строк после explode: ≈34.7 млн
1.11 Числовой аналог Вердикт: int64


## Блок 2. Жизненный цикл карточки инцидента

In [16]:
# Вопросы 2.1 - 2.3: Вердикты и Фазы
print("2.1 Распределение Вердикт:\n", df['Вердикт'].value_counts(dropna=False))
print("2.2 Доля FP:", (df['Вердикт'] == 'False Positive').mean())
print("2.3 Фазы реагирования:\n", df['Фаза реагирования'].value_counts(dropna=False))

# Вопросы 2.4 - 2.5: Комбинированные фазы (ИСПРАВЛЕНО: ищем именно те, что содержат разделитель)
phases = df['Фаза реагирования'].value_counts()
# Фильтруем только те фазы, которые содержат разделитель мультизначного поля
combined_phases = phases[phases.index.astype(str).str.contains('~~\|\|~~', na=False)]
print("2.5 Самая частая комбинированная фаза:", combined_phases.idxmax(), "-", combined_phases.max())

# Вопросы 2.6 - 2.8: Статусы, Спам, Эскалация (ИСПРАВЛЕНО: точное имя колонки из вашего списка)
print("2.6 Статусы SIEM:\n", df['Статус инцидента из SIEM'].value_counts(dropna=False))
print("2.7 Спам (True):", (df['Спам'] == 'True').sum())
print("2.8 Эскалация на L2 (True):", (df['Эскалировался на L2?'] == 'True').sum())

# Вопросы 2.9 - 2.12: Проверка на FP, ГИБ
print("2.9 Проверка на FP:\n", df['Ложное срабатывание'].value_counts(dropna=False))

# 2.10 Согласование Вердикта и флага
ct = pd.crosstab(df['Вердикт'], df['Ложное срабатывание'], dropna=False)
print("2.10 Таблица сопряженности:\n", ct)

print("2.11 Инцидент был изменен (True):", (df['Инцидент был изменен'] == 'True').sum())

# 2.12 ГИБ
gib_fp = df[df['Группа устранения'] == 'ГИБ']['Вердикт'].value_counts(normalize=True)
print("2.12 Доля FP в ГИБ:\n", gib_fp)

2.1 Распределение Вердикт:
 Вердикт
False Positive                       150675
Не указан                             54870
Решён                                 28606
Дубликат                               5819
Не решён                                316
Закрыт по совокупности инцидентов         8
Закрыт из родительской атаки              1
Name: count, dtype: int64
2.2 Доля FP: 0.6270417611685637
2.3 Фазы реагирования:
 Фаза реагирования
Закрыт                         239557
Закрыт~~||~~Проанализирован       519
Ожидание                          103
Закрыт~~||~~Сдерживание            55
В работе                           53
Классификация~~||~~Ожидание         4
Новый                               2
Устранение                          1
В работе~~||~~Классификация         1
Name: count, dtype: int64
2.5 Самая частая комбинированная фаза: Закрыт~~||~~Проанализирован - 519
2.6 Статусы SIEM:
 Статус инцидента из SIEM
New         239248
Closed         842
None           166
Approved      

## Блок 3. Кто и откуда: источники, аналитики, пользователи

In [8]:
# Вопросы 3.1 - 3.6: Пользователи и хосты
print("3.1 Уникальных suser:", df['suser'].nunique())
print("3.2 Уникальных жертв (Полное имя):", df['Полное имя'].nunique())
print("3.3 Топ жертв:\n", df['Полное имя'].value_counts().head(3))
print("3.4 Топ suser:\n", df['suser'].value_counts().head(3))
print("3.5 Уникальных хостов-целей:", df['Hostname Цели'].nunique())
print("3.6 Топ хостов-целей:\n", df['Hostname Цели'].value_counts().head(3))

# Вопросы 3.7 - 3.10: Аналитики и группы
print("3.7 Уникальных аналитиков:", df['Ответственный'].nunique())
print("3.9 Группы устранения:\n", df['Группа устранения'].value_counts())
print("3.10 Конвейер SIEM:\n", df['Конвейер SIEM'].value_counts(dropna=False))

3.1 Уникальных suser: 3971
3.2 Уникальных жертв (Полное имя): 1296
3.3 Топ жертв:
 Полное имя
Щукин Александр Евгеньевич     776
Филатов Владимир Николаевич    391
Мишин Никита Владимирович      269
Name: count, dtype: int64
3.4 Топ suser:
 suser
\root             1579
\ankey            1424
\administrator    1378
Name: count, dtype: int64
3.5 Уникальных хостов-целей: 8689
3.6 Топ хостов-целей:
 Hostname Цели
0.0.0.0\\                         4294
\srvSCCM03.iaivd.lan\srvSCCM03    1548
\srvSCCM04.iaivd.lan\srvSCCM04    1541
Name: count, dtype: int64
3.7 Уникальных аналитиков: 31
3.9 Группы устранения:
 Группа устранения
L1     236902
L2       3312
ГИБ        44
Name: count, dtype: int64
3.10 Конвейер SIEM:
 Конвейер SIEM
ООО "ИАиВД"                               128190
None                                       67665
srv-conv1.blinow-a12.iaivd.ru              19351
ГРО ЛО                                      9493
srv-conv02.blinow-a12.iaivd.ru              6823
DataGile                

## Блок 4. Время: профили активности, MTTR, SLA

In [9]:
# Преобразование времени
df['Создан_dt'] = pd.to_datetime(df['Создан'], errors='coerce')

# Вопросы 4.1 - 4.5: Период и профили
print("4.1 Период:", df['Создан_dt'].min(), "—", df['Создан_dt'].max())
print("4.2 Топ месяц:\n", df['Создан_dt'].dt.to_period('M').value_counts().head(1))
print("4.3 Мин месяц (полный):\n", df['Создан_dt'].dt.to_period('M').value_counts().tail(1))

hours = df['Создан_dt'].dt.hour.value_counts().sort_index()
print(f"4.4 Час пика: {hours.idxmax()} ({hours.max()}), минимум: {hours.idxmin()} ({hours.min()})")

weekends = df['Создан_dt'].dt.dayofweek.isin([5, 6]).mean()
print(f"4.5 Доля выходных: {weekends:.1%}")

# Вопросы 4.6 - 4.10: SLA и MTTR (предполагаем, что время в секундах)
# Примечание: замените 'Время решения' на реальное название колонки, если оно отличается
time_col = 'Время решения' 
median_res = df[time_col].median() / 3600
p90_res = df[time_col].quantile(0.9) / 3600
print(f"4.6 Медиана решения: {median_res:.1f} ч")
print(f"4.7 p90 решения: {p90_res:.1f} ч")
print(f"4.8 Доля >30 дней: {(df[time_col] > 30*86400).mean():.1%}")

print("4.9 Медиана по группам (ч):\n", df.groupby('Группа устранения')[time_col].median() / 3600)
print(f"4.10 Медиана взятия в работу: {df['Период взятия в работу'].median() / 3600:.1f} ч")

# Вопросы 4.11 - 4.12: Ловушки
# 4.11 Задержка перед автоназначением
ts_median = df['задержка перед автоназначением'].median()
print(f"4.11 Unix-timestamp автоназначения: {pd.to_datetime(ts_median, unit='s')}")

# 4.12 Единицы SLA
sla_median = df['SLA по устранению Атаки'].median()
print(f"4.12 SLA в мс: {sla_median} мс = {sla_median / 3.6e6} ч")

4.1 Период: 2024-04-12 17:06:04.854999+00:00 — 2025-09-03 12:21:17.348999+00:00
4.2 Топ месяц:
 Создан_dt
2024-06    37891
Freq: M, Name: count, dtype: int64
4.3 Мин месяц (полный):
 Создан_dt
2024-04    1
Freq: M, Name: count, dtype: int64
4.4 Час пика: 8 (19118), минимум: 0 (3591)
4.5 Доля выходных: 12.8%
4.6 Медиана решения: 35.7 ч
4.7 p90 решения: 8858.6 ч
4.8 Доля >30 дней: 17.1%
4.9 Медиана по группам (ч):
 Группа устранения
L1        33.858717
L2      8076.048719
ГИБ    40107.871820
Name: Время решения, dtype: float64
4.10 Медиана взятия в работу: 24.5 ч
4.11 Unix-timestamp автоназначения: 2024-10-17 05:32:03
4.12 SLA в мс: 3600000.0 мс = 1.0 ч


## Блок 5. Содержательный анализ: процессы, теги, пары

In [10]:
# Вопросы 5.1 - 5.5: Процессы и теги
print("5.1 Топ процессов:\n", df['processName'].value_counts().head(8))
ps_share = df['Команда'].dropna().str.contains('powershell', case=False).mean()
print(f"5.2 Доля powershell: {ps_share:.1%}")
print("5.3 Топ Пиктограмма IRP:\n", df['Пиктограмма IRP'].value_counts().head(5))
print("5.4 Топ Теги:\n", df['Теги'].value_counts().head(3))
comb_tags = df['Теги'].dropna().str.contains('~~||~~').mean()
print(f"5.5 Доля комбинированных тегов: {comb_tags:.1%}")

# Вопросы 5.6 - 5.8: Разбор пар "Значение Инцидента (группировка)"
col_pairs = 'Значение Инцидента (группировка)'
pairs_exploded = df[col_pairs].dropna().str.split('~~||~~').explode()
pairs_split = pairs_exploded.str.split('|', n=1, expand=True)
pairs_split.columns = ['key', 'value']

print("5.6 Роли (значения):\n", pairs_split['value'].value_counts())
print("5.7 Топ ключей:\n", pairs_split['key'].value_counts().head(3))
print(f"5.8 Всего пар: {len(pairs_split)}, уникальных ключей: {pairs_split['key'].nunique()}")

# Вопросы 5.9 - 5.12: Доли и статистики
print("5.9 Доля FP по критичности:\n", df.groupby('Критичность')['Вердикт'].apply(lambda x: (x=='False Positive').mean()))
print("5.10 Доля FP по группам:\n", df.groupby('Группа устранения')['Вердикт'].apply(lambda x: (x=='False Positive').mean()))

print("5.11 Конвейер srv-conv1:\n", df[df['Конвейер SIEM'].str.contains('srv-conv1', na=False)].shape[0])

lens = df[col_pairs].dropna().str.split('~~||~~').str.len()
print(f"5.12 Среднее элементов: {lens.mean():.0f}, медиана: {lens.median():.0f}")

5.1 Топ процессов:
 processName
reg.exe         4030
msiexec.exe     3342
ipconfig.exe    3100
rundll32.exe    2305
regsvr32.exe    2007
sc.exe          1975
ssh.exe         1659
cmd.exe         1477
Name: count, dtype: int64
5.2 Доля powershell: 5.3%
5.3 Топ Пиктограмма IRP:
 Пиктограмма IRP
Подозрительная активность    93587
Управление УЗ                41255
Сетевая безопасность         36979
Атака на инфраструтуру       24446
Инцидент                     17615
Name: count, dtype: int64
5.4 Топ Теги:
 Теги
discovery                                                                               17176
malware                                                                                  9193
account compromised~~||~~credential access~~||~~initial access~~||~~lateral movement     3046
Name: count, dtype: int64
5.5 Доля комбинированных тегов: 100.0%
5.6 Роли (значения):
 value
    3277124
Name: count, dtype: int64
5.7 Топ ключей:
 key
     5828830
.    2989693
1    2051147
Name: count,

## Блок 6. Качество данных

In [11]:
# Вопросы 6.1 - 6.5: Дубликаты, пропуски, NaN vs Значения
print("6.1 Дубликатов EntityId:", df['EntityId'].duplicated().sum())
empty_comments = df['Комментарий'].fillna("").eq("").sum()
print("6.2 Пустых комментариев:", empty_comments)
print("6.3 Заполнено поле Ошибки:", df['Ошибки'].notna().sum())
print("6.4 NaN у Эскалировался на L2?:", df['Эскалировался на L2?'].isna().sum())
print("6.5 Вердикт NaN:", df['Вердикт'].isna().sum(), "| 'Не указан':", (df['Вердикт'] == 'Не указан').sum())

# Вопросы 6.6 - 6.8: Регистры и выбросы
print("6.6 Примеры разного регистра в suser:", df['suser'].str.lower().value_counts().head(3))
max_hours = df['Время решения'].max() / 3600
print(f"6.7 Макс Время решения: {max_hours:,.1f} ч (технический выброс)")
print(f"6.8 SLA медиана: {df['SLA по устранению Атаки'].median() / 3.6e6} ч")

6.1 Дубликатов EntityId: 0
6.2 Пустых комментариев: 69156
6.3 Заполнено поле Ошибки: 13
6.4 NaN у Эскалировался на L2?: 238539
6.5 Вердикт NaN: 0 | 'Не указан': 54870
6.6 Примеры разного регистра в suser: suser
\root             1579
\ankey            1424
\administrator    1378
Name: count, dtype: int64
6.7 Макс Время решения: 487,521,350.8 ч (технический выброс)
6.8 SLA медиана: 1.0 ч


## Блок 7. Мост к ML

In [12]:
# Вопросы 7.1 - 7.10: Подготовка к моделированию
# 7.1 Баланс классов
df['y'] = (df['Вердикт'] == 'False Positive').astype(int)
print(f"7.1 Баланс FP: {df['y'].mean():.3f}")

# 7.3 Высокая кардинальность
high_card = df.nunique().sort_values(ascending=False).head(4)
print("7.3 Поля с высокой кардинальностью:\n", high_card)

# 7.4 Числовые признаки
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("7.4 Числовые признаки:", num_cols)

# 7.6 Утечки
print("7.6 Потенциальные утечки: флаг 'Ложное срабатывание', текст 'Решение'")

# 7.7 Булевые признаки
bool_cols = [c for c in df.columns if df[c].isin(['True', 'False']).any()]
print(f"7.7 Найдено {len(bool_cols)} булевых признаков")

# 7.10 Идентификаторы
print("7.10 Исключаем: EntityId, verdict (дубликат)")

7.1 Баланс FP: 0.627
7.3 Поля с высокой кардинальностью:
 EntityId                240295
Инцидент в Источнике    221038
Принятые меры           214105
Время решения           202832
dtype: int64
7.4 Числовые признаки: ['EntityId', 'Ответсвенный по РП', 'Количество инцидентов', 'Время решения', 'SLA по устранению Атаки', 'SLA по взятию в работу Атаки', 'Период взятия в работу', 'задержка перед автоназначением', 'verdict', 'y']
7.6 Потенциальные утечки: флаг 'Ложное срабатывание', текст 'Решение'
7.7 Найдено 18 булевых признаков
7.10 Исключаем: EntityId, verdict (дубликат)
